# N-CMAPSS DS05 v10 — does the CNN branch earn its place?

## What v9's winner actually said

The aggregate boundary test flagged only `f1`, because comparing the top twelve collectively washes
out a corner point. The selected configuration itself sits at a wall in **8 of 13 dimensions**:

| parameter | value | position |
|---|---|---|
| n_conv | 2 | minimum |
| f1 | 192 | maximum |
| f_growth | 0.50 | minimum |
| n_lstm | 1 | minimum |
| dense1 | 8 | minimum |
| dense2 | 0 | minimum |
| lr | 0.005 | maximum |
| batch | 64 | minimum |

The other new winners agree: `new3_PSO` and `new4_PSO` both use n_conv=2, f_growth 0.65-0.68,
dense1 under 27, dense2 under 11, high learning rate, no pooling. (`new5_PSO` is v8_cand2
rediscovered — it was seeded as a particle, so it does not count as an independent find.)

Read as a description rather than a list of numbers, the search is asking for a wide first
convolution that shrinks as fast as the range allows, as few conv layers as permitted, no pooling,
and essentially no dense head — trained at the highest learning rate and smallest batch on offer.

**It is trying to delete the convolutional branch and the dense head, and keeps hitting the floor of
what the parameterisation allows it to remove.**

## So this notebook lets it

**1. A widened box, searched by PSO.** Every wall the v9 winner touched is moved: n_conv down to 0,
f1 up to 384, f_growth down to 0.25, dense1 and dense2 down to 0, lr up to 2e-2, batch down to 32.
PSO only — BO lost to random search at equal budget in both v8 and v9, so a third replication is not
worth 90 minutes of GPU. The v9 winners seed two particles.

**2. A branch ablation at the winning configuration.** Full hybrid against LSTM-only, CNN-only and
no-dense-head, on the same six seeds, with paired tests. This is the question that matters for a
thesis about hybrid CNN-LSTM architectures, and it deserves a direct experiment rather than an
inference from where the optimiser drifted.

Either outcome is reportable. On FD003 architecture was the largest single effect measured anywhere
in this project — 7.07 RMSE, far above the best hyperparameter at 0.51. If DS05 says the opposite,
that contrast is a finding about the datasets, not a failure.

**3. Finally, the uncapped raw number.** The 28-sensor/T=30 representation at 1-in-10 sampling with
capping switched off, using whatever wins above. v7 put that cell at 8.112 raw using a fifth of the
data, against our 8.247 thesis figure, so this is the defensible raw result that is not handicapped
by capped training.

## Discipline, unchanged

One budget everywhere. Test recorded every trial and never returned to the optimiser. Two-stage
selection with a paired t-test on per-seed differences. Both target conventions in every table.

In [ ]:
import os, json, gc, time, random, math, warnings
os.environ['PYTHONHASHSEED']='42'
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, h5py
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr, ttest_rel, mannwhitneyu
from scipy.stats.qmc import Sobol
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Concatenate, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def set_seeds(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s)
    tf.keras.utils.set_random_seed(s)

def preflight():
    if not tf.config.list_physical_devices('GPU'): print('CPU mode'); return False
    try:
        x=np.random.rand(8,10,3).astype('float32'); y=np.random.rand(8,1).astype('float32')
        i=Input(shape=(10,3)); o=Dense(1)(GlobalAveragePooling1D()(Conv1D(4,3,padding='same')(i)))
        m=Model(i,o); m.compile('adam','mse'); m.fit(x,y,epochs=1,verbose=0); K.clear_session()
        print('GPU preflight passed'); return True
    except Exception as e:
        print('GPU preflight FAILED (%s)'%type(e).__name__); K.clear_session(); return False

USE_GPU=preflight(); DEVICE='/GPU:0' if USE_GPU else '/CPU:0'
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)
print('TF',tf.__version__,'| device',DEVICE)

## Data — identical to v7/v8/v9 so every number stays comparable

In [ ]:
SEED=42; SUB=50; VAL_UNITS=[5,6]; CAP=55; T=30
EPOCHS=100; PATIENCE=15
N_PSO=36; SWARM=9; N_SEEDS2=6; SHORTLIST=2
set_seeds(SEED)

def read_sub(dset,step,dtype=np.float32,chunk=400_000):
    n=dset.shape[0]; out=[]
    for a in range(0,n,chunk):
        b=min(a+chunk,n); blk=dset[a:b]; first=(-a)%step
        if first<len(blk): out.append(blk[first::step].astype(dtype,copy=False))
    return np.concatenate(out)

def windows(F,u,c,y,mask,T):
    uu,cc,FF,yy=u[mask],c[mask],F[mask],y[mask]
    key=uu.astype(np.int64)*100000+cc; o=np.argsort(key,kind='stable')
    FF,yy,key=FF[o],yy[o],key[o]; bnd=np.flatnonzero(np.diff(key))+1
    X,Y=[],[]
    for a,b in zip(np.r_[0,bnd],np.r_[bnd,len(key)]):
        A=FF[a:b]; L=yy[a:b]
        for e in range(T,len(A)+1): X.append(A[e-T:e]); Y.append(L[e-1])
    return np.asarray(X,np.float32),np.asarray(Y,np.float32)

def build_dataset(sub, cap):
    with h5py.File('N-CMAPSS_DS05.h5','r') as h:
        Sd=read_sub(h['X_s_dev'],sub); Vd=read_sub(h['X_v_dev'],sub)
        Ad=read_sub(h['A_dev'],sub);   Yd=read_sub(h['Y_dev'],sub)
        St=read_sub(h['X_s_test'],sub);Vt=read_sub(h['X_v_test'],sub)
        At=read_sub(h['A_test'],sub);  Yt=read_sub(h['Y_test'],sub)
    Fd=np.hstack([Sd,Vd]); Ft=np.hstack([St,Vt])
    ud=Ad[:,0].astype(int); cd=Ad[:,1].astype(int); yd=Yd.ravel().astype(np.float32)
    ut=At[:,0].astype(int); ct=At[:,1].astype(int); yt=Yt.ravel().astype(np.float32)
    del Sd,Vd,Ad,Yd,St,Vt,At,Yt; gc.collect()
    tr=~np.isin(ud,VAL_UNITS); va=np.isin(ud,VAL_UNITS)
    sc=MinMaxScaler().fit(Fd[tr])
    Fd2=sc.transform(Fd).astype(np.float32); Ft2=sc.transform(Ft).astype(np.float32)
    Xtr,ytr_raw=windows(Fd2,ud,cd,yd,tr,T)
    Xva,yva_raw=windows(Fd2,ud,cd,yd,va,T)
    Xte,yte_raw=windows(Ft2,ut,ct,yt,np.ones(len(yt),bool),T)
    del Fd,Ft,Fd2,Ft2; gc.collect()
    if cap:
        ytr=np.minimum(ytr_raw,CAP); yva=np.minimum(yva_raw,CAP)
    else:
        ytr=ytr_raw; yva=yva_raw
    d=dict(Xtr=Xtr,ytr=ytr,Xva=Xva,yva=yva,Xte=Xte,yte_raw=yte_raw,
           yte_cap=np.minimum(yte_raw,CAP),shape=(T,Xtr.shape[2]),
           units_train=sorted(set(ud[tr])))
    d['triv_raw']=float(np.sqrt(mean_squared_error(yte_raw,np.full_like(yte_raw,ytr_raw.mean()))))
    d['triv_cap']=float(np.sqrt(mean_squared_error(d['yte_cap'],
                        np.full_like(d['yte_cap'],np.minimum(ytr_raw,CAP).mean()))))
    return d

DS=build_dataset(SUB, cap=True)
SHAPE=DS['shape']
TRIV_CAP=DS['triv_cap']; TRIV_RAW=DS['triv_raw']
print('shape %s | train %s | val %s | test %s'%(SHAPE,format(len(DS['ytr']),','),
      format(len(DS['yva']),','),format(len(DS['yte_raw']),',')))
print('trivial floors -> raw %.3f | capped-%d %.3f'%(TRIV_RAW,CAP,TRIV_CAP))
print('leakage check: val units %s vs train units %s'%(VAL_UNITS,DS['units_train']))

## The widened box

Every wall the v9 winner touched is moved outward. `n_conv=0` and `dense1=0` are now legal values,
so the search can remove the convolutional branch and the dense head on its own rather than pressing
against a floor that stops it.

In [ ]:
SPACE=[('n_conv','int',0,4),          # was 2-4     -- winner at the floor
       ('f1','int',24,384),           # was 24-192  -- winner at the ceiling
       ('f_growth','float',0.25,2.0), # was 0.5-2.0 -- winner at the floor
       ('kernel','cat',[1,2,3,5,7]),
       ('use_pool','cat',[0,1]),
       ('n_lstm','int',1,2),
       ('lstm_units','int',24,320),
       ('dropout','float',0.0,0.5),   # was 0-0.4
       ('l2reg','log',1e-7,1e-1),
       ('dense1','int',0,256),        # was 8-256   -- winner at the floor
       ('dense2','int',0,256),
       ('lr','log',1e-4,2e-2),        # was 1e-4-5e-3 -- winner at the ceiling
       ('batch','cat',[32,64,128,256,512,1024])]
D=len(SPACE); HPN=[s[0] for s in SPACE]

def decode(z):
    z=np.clip(np.asarray(z,float),0,1); out={}
    for zi,s in zip(z,SPACE):
        n,k=s[0],s[1]
        if k=='int':    out[n]=int(round(s[2]+zi*(s[3]-s[2])))
        elif k=='float':out[n]=float(s[2]+zi*(s[3]-s[2]))
        elif k=='log':  out[n]=float(np.exp(np.log(s[2])+zi*(np.log(s[3])-np.log(s[2]))))
        else:           out[n]=s[2][min(int(zi*len(s[2])),len(s[2])-1)]
    out['use_lstm']=1
    return out

def encode(kw):
    z=[]
    for s in SPACE:
        n,k=s[0],s[1]; v=kw[n]
        if k in ('int','float'): z.append((v-s[2])/(s[3]-s[2]))
        elif k=='log':           z.append((np.log(v)-np.log(s[2]))/(np.log(s[3])-np.log(s[2])))
        else:                    z.append((s[2].index(v)+0.5)/len(s[2]))
    return np.clip(np.array(z,float),0,1)

def nrmsd(a,b): return float(np.linalg.norm(np.asarray(a)-np.asarray(b))/np.sqrt(D))

V9_WIN=dict(n_conv=2,f1=192,f_growth=0.5,kernel=5,use_pool=0,n_lstm=1,lstm_units=88,
            dropout=0.3442517771420791,l2reg=3.831223699431197e-05,dense1=8,dense2=0,
            lr=0.005000000000000002,batch=64,use_lstm=1)
V9_ALT=dict(n_conv=2,f1=152,f_growth=0.679117,kernel=3,use_pool=0,n_lstm=2,lstm_units=69,
            dropout=0.18910247844631278,l2reg=2.5767391470093876e-05,dense1=13,dense2=6,
            lr=0.002510206810004858,batch=128,use_lstm=1)
KNOWN=[('v9_win',V9_WIN),('v9_alt',V9_ALT)]
Z_KNOWN=[encode(k[1]) for k in KNOWN]

for nm,kw in KNOWN:
    d=decode(encode(kw))
    bad=[k for k in kw if k!='use_lstm' and d[k]!=kw[k] and abs(d[k]-kw[k])>1e-4*max(1,abs(kw[k]))]
    print('%-8s round-trip %s'%(nm,'OK' if not bad else 'MISMATCH '+str(bad)))
print('dimension',D)
print('v9 winner position in the NEW box (0=floor, 1=ceiling):')
for n,v in zip(HPN,encode(V9_WIN)): print('   %-12s %.3f'%(n,v))

In [ ]:
def build(kw,shape=SHAPE,seed=SEED):
    # n_conv=0 removes the convolutional branch; use_lstm=0 removes the recurrent one.
    reg=regularizers.l2(kw['l2reg']) if kw['l2reg']>1e-6 else None
    kk=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),kernel_regularizer=reg)
    inp=Input(shape=shape); branches=[]
    if kw['n_conv']>0:
        c=inp
        for i in range(kw['n_conv']):
            f=max(8,int(round(kw['f1']*(kw['f_growth']**i))))
            c=Conv1D(f,kw['kernel'],padding='same',activation='relu',**kk)(c)
            if kw['dropout']>0: c=Dropout(kw['dropout'],seed=seed)(c)
            if kw['use_pool'] and i<2 and int(c.shape[1])>2: c=MaxPooling1D(2,padding='same')(c)
        branches.append(GlobalAveragePooling1D()(c))
    if kw.get('use_lstm',1):
        r=inp
        for j in range(kw['n_lstm']):
            r=LSTM(kw['lstm_units'],return_sequences=(j<kw['n_lstm']-1),**kk)(r)
        branches.append(r)
    if not branches: raise ValueError('no branch enabled')
    h=branches[0] if len(branches)==1 else Concatenate()(branches)
    if kw['dense1']>=8: h=Dense(kw['dense1'],activation='relu',**kk)(h)
    if kw['dense2']>=16: h=Dense(kw['dense2'],activation='relu',**kk)(h)
    m=Model(inp,Dense(1,activation='linear',**kk)(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(kw['lr']),loss='mse'); return m

TRIALS=[]
def evaluate(kw,seed=SEED,tag='',ds=None,epochs=None,patience=None):
    # returns VALIDATION only; test is recorded and never fed back to any optimiser
    d=DS if ds is None else ds
    ep=EPOCHS if epochs is None else epochs
    pa=PATIENCE if patience is None else patience
    K.clear_session(); gc.collect(); set_seeds(seed); t0=time.time()
    try:
        with tf.device(DEVICE):
            m=build(kw,shape=d['shape'],seed=seed)
            m.fit(d['Xtr'],d['ytr'],validation_data=(d['Xva'],d['yva']),epochs=ep,
                  batch_size=kw['batch'],verbose=0,
                  callbacks=[EarlyStopping('val_loss',patience=pa,restore_best_weights=True),
                             ReduceLROnPlateau('val_loss',factor=0.5,patience=max(2,pa//2),min_lr=1e-6)])
            pv=m.predict(d['Xva'],batch_size=8192,verbose=0).ravel()
            pt=m.predict(d['Xte'],batch_size=8192,verbose=0).ravel()
        val=float(np.sqrt(mean_squared_error(d['yva'],pv)))
        tc =float(np.sqrt(mean_squared_error(d['yte_cap'],np.clip(pt,0,CAP))))
        trw=float(np.sqrt(mean_squared_error(d['yte_raw'],pt)))
        npm=int(m.count_params())
    except Exception as e:
        print('   trial failed (%s) -> penalised'%type(e).__name__)
        val,tc,trw,npm=1e3,np.nan,np.nan,0
    if not np.isfinite(val): val=1e3
    rec=dict(tag=tag,seed=seed,val=val,test_capped=tc,test_raw=trw,params=npm,
             secs=round(time.time()-t0,1))
    rec.update({k:v for k,v in kw.items()})
    TRIALS.append(rec); return val

def far_design(n,refs,min_d=0.35,seed=SEED):
    eng=Sobol(d=D,scramble=True,seed=seed); pts=[]; guard=0
    while len(pts)<n and guard<50:
        guard+=1
        for c in eng.random(max(8,n*4)):
            if min(nrmsd(c,r) for r in refs)>=min_d:
                pts.append(c)
                if len(pts)==n: break
    rs=np.random.RandomState(seed)
    while len(pts)<n:
        c=rs.rand(D)
        if min(nrmsd(c,r) for r in refs)>=min_d: pts.append(c)
    return np.array(pts[:n])

print('warm-up: the two v9 winners, one seed each')
KNOWN_Y={}; KNOWN_REC={}
for nm,kw in KNOWN:
    KNOWN_Y[nm]=evaluate(kw,SEED,'known:'+nm); r=TRIALS[-1]; KNOWN_REC[nm]=dict(r)
    print('  %-8s val %6.3f | capped %6.3f | raw %7.3f | %8d params | %4.0fs'
          %(nm,r['val'],r['test_capped'],r['test_raw'],r['params'],r['secs']))
_s=float(np.mean([t['secs'] for t in TRIALS]))
print('estimated: %.0f min for %d PSO evaluations'%(N_PSO*_s/60,N_PSO))

## PSO on the widened box

Particles 0 and 1 are the two v9 winners, so the swarm cannot return something worse than we already
have. The rest start at least 0.35 normalised distance away. BO is not run: it lost to random search
at equal budget in v8 and in v9, and a third replication buys nothing the paper does not already have.

In [ ]:
ITERS=N_PSO//SWARM
rng=np.random.RandomState(SEED+7)
P=far_design(SWARM,Z_KNOWN,seed=SEED+7)
for i,zk in enumerate(Z_KNOWN): P[i]=zk.copy()
V=(rng.rand(SWARM,D)-0.5)*0.30
pbest=P.copy(); pbest_y=np.full(SWARM,np.inf); gbest=None; gbest_y=np.inf
w_hi,w_lo,c1,c2=0.85,0.35,1.5,1.5
t0=time.time()
for it in range(ITERS):
    w_in=w_hi-(w_hi-w_lo)*it/max(1,ITERS-1)
    for k in range(SWARM):
        if it==0 and k<len(KNOWN):
            nm=KNOWN[k][0]; y=KNOWN_Y[nm]
            rec=dict(KNOWN_REC[nm]); rec['tag']='pso'; TRIALS.append(rec)
        else:
            y=evaluate(decode(P[k]),SEED,'pso')
        if y<pbest_y[k]: pbest_y[k]=y; pbest[k]=P[k].copy()
        if y<gbest_y:    gbest_y=y;    gbest=P[k].copy()
    r1,r2=rng.rand(SWARM,D),rng.rand(SWARM,D)
    V=np.clip(w_in*V+c1*r1*(pbest-P)+c2*r2*(gbest-P),-0.25,0.25); P=np.clip(P+V,0,1)
    print('  pso iter %d/%d  gbest %7.3f  spread %.3f'
          %(it+1,ITERS,gbest_y,float(np.mean(np.std(P,axis=0)))))
print('PSO done in %.1f min | best val %.3f'%((time.time()-t0)/60,gbest_y))
print('v9 reference: PSO best val 3.615 | v9 winner stage-2 val 3.797')
GB=decode(gbest)
print('gbest configuration:')
for n in HPN: print('   %-12s %s'%(n,GB[n]))

## The branch ablation

Four variants of one configuration, same six seeds, paired tests. `LSTM only` sets n_conv to 0;
`CNN only` disables the recurrent branch; `no dense head` removes both dense layers. If the full
hybrid does not beat `LSTM only` at p<0.05, the convolutional branch is not earning its place on this
dataset and the paper should say so.

In [ ]:
ABL_BASE=dict(GB)
SEEDS_A=[SEED+300+s for s in range(N_SEEDS2)]
VARIANTS=[('full hybrid',   dict(ABL_BASE)),
          ('LSTM only',     dict(ABL_BASE, n_conv=0)),
          ('CNN only',      dict(ABL_BASE, use_lstm=0)),
          ('no dense head', dict(ABL_BASE, dense1=0, dense2=0))]
VAR_KW={nm:kw for nm,kw in VARIANTS}
if ABL_BASE['n_conv']==0:
    print('NOTE: the search itself chose n_conv=0, so full hybrid and LSTM only coincide.')
AB={}; rows=[]
for nm,kw in VARIANTS:
    if nm=='CNN only' and kw['n_conv']==0:
        print('  %-16s skipped (no branch would remain)'%nm); continue
    v,c,r=[],[],[]
    for s in SEEDS_A:
        evaluate(kw,s,'abl:'+nm); t=TRIALS[-1]
        v.append(t['val']); c.append(t['test_capped']); r.append(t['test_raw'])
    AB[nm]=dict(val=np.array(v),capped=np.array(c),raw=np.array(r))
    rows.append(dict(variant=nm,val=float(np.mean(v)),capped=float(np.mean(c)),
                     capped_sd=float(np.std(c,ddof=1)),raw=float(np.mean(r)),
                     raw_sd=float(np.std(r,ddof=1)),params=int(t['params'])))
    print('  %-16s val %6.3f | capped %6.3f +/- %.3f | raw %7.3f | %8d params'
          %(nm,np.mean(v),np.mean(c),np.std(c,ddof=1),np.mean(r),t['params']))
A=pd.DataFrame(rows)
print()
print('paired tests against the full hybrid, same %d seeds:'%N_SEEDS2)
for nm in AB:
    if nm=='full hybrid': continue
    d=AB[nm]['capped']-AB['full hybrid']['capped']
    t,p=ttest_rel(AB[nm]['capped'],AB['full hybrid']['capped'])
    verdict='worse than the hybrid' if d.mean()>0 else 'better than the hybrid'
    print('  %-16s diff %+7.3f  SE %.3f  t %+6.2f  p %.4f  %s%s'
          %(nm,d.mean(),d.std(ddof=1)/math.sqrt(len(d)),t,p,verdict,
            '  (SIGNIFICANT)' if p<0.05 else '  (within noise)'))
print()
if 'LSTM only' in AB:
    d=AB['LSTM only']['capped']-AB['full hybrid']['capped']
    _,p=ttest_rel(AB['LSTM only']['capped'],AB['full hybrid']['capped'])
    if p<0.05 and d.mean()>0:
        print('  => the convolutional branch EARNS its place on DS05')
    elif p<0.05 and d.mean()<0:
        print('  => the convolutional branch HURTS on DS05')
    else:
        print('  => the convolutional branch is NOT distinguishable from removing it (n=%d)'%N_SEEDS2)
A.to_csv('ncmapss_v10_ablation.csv',index=False)

## Stage 2 — head-to-head against the v9 incumbent

In [ ]:
SR=pd.DataFrame(TRIALS)
SR=SR[(SR.tag=='pso')&(SR.val<999)]
short=SR.nsmallest(SHORTLIST*3,'val').drop_duplicates(subset=HPN).head(SHORTLIST)
print('shortlist by stage-1 validation (test shown, not used to choose):')
print(short[['val','test_capped','test_raw','n_conv','f1','dense1','lr','batch','params']].to_string(index=False))

def kw_of(row):
    g=(lambda n: getattr(row,n)) if hasattr(row,'_fields') else (lambda n: row[n])
    out={}
    for s in SPACE:
        n,k=s[0],s[1]; v=g(n)
        out[n]=int(v) if (k=='int' or n in ('kernel','use_pool','batch')) else float(v)
    out['use_lstm']=1
    return out

SEEDS2=[SEED+400+s for s in range(N_SEEDS2)]
PER={}; STAGE2=[]; CAND_KW={}
cands=[('v9_win',dict(V9_WIN))]+[('v10_%d'%(i+1),kw_of(r)) for i,r in enumerate(short.itertuples())]
best_abl=A.nsmallest(1,'capped').iloc[0]['variant'] if len(A) else None
if best_abl and best_abl!='full hybrid':
    cands.append(('abl_'+best_abl.replace(' ','_'),dict(VAR_KW[best_abl])))
for name,kw in cands:
    CAND_KW[name]=dict(kw)
    v,c,r=[],[],[]
    for s in SEEDS2:
        evaluate(kw,s,'stage2:'+name); t=TRIALS[-1]
        v.append(t['val']); c.append(t['test_capped']); r.append(t['test_raw'])
    PER[name]=dict(val=np.array(v),capped=np.array(c),raw=np.array(r))
    STAGE2.append(dict(name=name,val=float(np.mean(v)),val_sd=float(np.std(v,ddof=1)),
                       capped=float(np.mean(c)),capped_sd=float(np.std(c,ddof=1)),
                       raw=float(np.mean(r)),raw_sd=float(np.std(r,ddof=1))))
    print('  %-16s val %6.3f+/-%.3f | capped %6.3f+/-%.3f | raw %7.3f+/-%.3f'
          %(name,np.mean(v),np.std(v,ddof=1),np.mean(c),np.std(c,ddof=1),np.mean(r),np.std(r,ddof=1)))
S2=pd.DataFrame(STAGE2)

win=S2[S2['name']!='v9_win'].nsmallest(1,'val').iloc[0]['name']
print()
print('validation selects: %s   (selection never saw a test score)'%win)
d=PER[win]['capped']-PER['v9_win']['capped']
tt,pp=ttest_rel(PER[win]['capped'],PER['v9_win']['capped'])
print('paired vs v9 incumbent: diff %+7.3f  SE %.3f  t %+6.2f  p %.4f'
      %(d.mean(),d.std(ddof=1)/math.sqrt(len(d)),tt,pp))
ADOPT=bool(d.mean()<0 and pp<0.05)
print('verdict:','ADOPT the v10 configuration' if ADOPT else 'KEEP v9 (difference within noise)')
FINAL_NAME = win if ADOPT else 'v9_win'
FINAL_KW = dict(CAND_KW[FINAL_NAME])
fin=S2[S2['name']==FINAL_NAME].iloc[0]
print()
print('%-40s %9s %9s %8s'%('','capped-55','raw','skill'))
for lab,cc,rr in [('MSTSDN published',3.950,None),
                  ('ours, v10',fin.capped,fin.raw),
                  ('ours, v9',4.859,10.038),
                  ('ours, v8',5.015,10.131),
                  ('ours, v7 matched protocol',5.137,10.268),
                  ('trivial floor',TRIV_CAP,TRIV_RAW)]:
    print('%-40s %9.3f %9s %+8.3f'%(lab,cc,('%.3f'%rr) if rr is not None else '  --  ',1-cc/TRIV_CAP))
print('remaining gap to MSTSDN: %+.3f'%(fin.capped-3.95))
S2.to_csv('ncmapss_v10_stage2.csv',index=False)
pd.DataFrame(TRIALS).to_csv('ncmapss_v10_all_trials.csv',index=False)

## The uncapped raw number at 1-in-10

The thesis quotes 8.247 raw for DS05, measured on the old 18-feature snapshot representation at
1-in-10 sampling. v7 put the 28-sensor/T=30 representation at 8.112 raw using a fifth of that data.
This cell runs the winning configuration on that representation at 1-in-10 with **capping switched
off**, which is the raw figure comparable to what is already written.

Two practical points, both of which have bitten this project before.

**Budget is matched in gradient updates, not epochs.** Five times the data means five times the
updates per epoch, so the epoch count is divided by five rather than held fixed. Holding epochs
constant would quietly give the big run five times the training of everything it is being compared
against — the mirror image of the 113x under-training we found in the original N-CMAPSS rebuild.

**Memory is roughly 2.3 GB of float32 windows.** If the machine cannot hold it the cell falls back to
1-in-20 and prints which sampling it used. A declared fallback is honest; a MemoryError halfway
through a three-hour run is not.

In [ ]:
n_small=len(DS['ytr'])
del DS; gc.collect()
SUB_BIG=10
try:
    DS10=build_dataset(SUB_BIG, cap=False)
except MemoryError:
    print('MemoryError at 1-in-%d -> falling back to 1-in-20'%SUB_BIG)
    SUB_BIG=20; gc.collect(); DS10=build_dataset(SUB_BIG, cap=False)

ratio=len(DS10['ytr'])/float(n_small)
EP_BIG=max(10,int(round(EPOCHS/ratio))); PA_BIG=max(4,int(round(PATIENCE/ratio)))
print('1-in-%d uncapped | train %s windows (%.1fx) | test %s | trivial raw %.3f'
      %(SUB_BIG,format(len(DS10['ytr']),','),ratio,format(len(DS10['yte_raw']),','),DS10['triv_raw']))
print('update-matched budget: %d epochs, patience %d (was %d / %d at 1-in-%d)'
      %(EP_BIG,PA_BIG,EPOCHS,PATIENCE,SUB))

vals,raws,caps=[],[],[]
for s in range(5):
    evaluate(FINAL_KW,SEED+500+s,'uncapped_1in%d'%SUB_BIG,ds=DS10,epochs=EP_BIG,patience=PA_BIG)
    t=TRIALS[-1]; vals.append(t['val']); raws.append(t['test_raw']); caps.append(t['test_capped'])
    print('  seed %d  raw %7.3f | capped %6.3f | %4.0fs'%(SEED+500+s,t['test_raw'],t['test_capped'],t['secs']))
raws=np.array(raws); caps=np.array(caps); fr=DS10['triv_raw']
print()
print('%-52s %9s %8s'%('','raw RMSE','skill'))
for lab,v in [('thesis figure, 18-feature snapshot at 1-in-10',8.247),
              ('v7, 28 sensors + T=30, uncapped, 1-in-50',8.112),
              ('v10, 28 sensors + T=30, uncapped, 1-in-%d'%SUB_BIG,float(raws.mean())),
              ('trivial floor',fr)]:
    print('%-52s %9.3f %+8.3f'%(lab,v,1-v/fr))
print('v10 uncapped: %.3f +/- %.3f over 5 seeds'%(raws.mean(),raws.std(ddof=1)))
print('(capped metric on the same runs, for reference: %.3f +/- %.3f)'%(caps.mean(),caps.std(ddof=1)))

json.dump(dict(final_name=FINAL_NAME,final_kw={k:float(v) for k,v in FINAL_KW.items()},
               adopt=ADOPT,ablation=A.to_dict('records'),
               v10_capped=float(fin.capped),v10_raw=float(fin.raw),
               uncapped_sub=SUB_BIG,uncapped_epochs=EP_BIG,
               uncapped_raw_mean=float(raws.mean()),uncapped_raw_sd=float(raws.std(ddof=1)),
               trivial_raw_big=float(fr),trivial_capped_small=float(TRIV_CAP)),
          open('ncmapss_v10_summary.json','w'),indent=2)
pd.DataFrame(TRIALS).to_csv('ncmapss_v10_all_trials.csv',index=False)
print()
print('saved ncmapss_v10_ablation.csv, _stage2.csv, _all_trials.csv, _summary.json in',os.getcwd())